**Попытка попробовать другую модель # 3**

На основе посчитанного ранее рейтинга и данных об оптимальной конфигурации ALS была предпринята еще одна попытка повышения качества предсказания в виде гибридной модели (матричная факторизация + NN с кастомным embedding-слоем). Результаты на Kaggle оказались чуть хуже простой ALS модели, поэтому в дальнейшем было принято решение немного сменить подход к предсказанию, о чем можно почитать в следующем ноутбуке.

In [1]:
import os
import csv
import pandas as pd
import numpy as np
import matplotlib
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import BaselineOnly
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from surprise.model_selection import cross_validate, PredefinedKFold, GridSearchCV
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from itertools import product
import tensorflow as tf

2025-02-28 22:23:57.414784: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-28 22:23:57.414827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-28 22:23:57.416005: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-28 22:23:57.420287: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-28 22:23:58.258056: W tensorflow/compiler/tf2

In [2]:
#Напишем базовую комплектацию показателей для проверки точности вычисления рейтинга моделью.
def Avg_Precision_at_n(fact, predicted, n=10):
    """
    Вычисление средней точности по n-позициям для двух списков значений.
    
    Вход
    ----------
    fact : list
             Фактический список элементов, который нужно предсказать.
    predicted : list
             Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
             Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Средняя точность на n-позициях
    """
    if not fact:
        return 0.0

    if len(predicted)>n:
        predicted = predicted[:n]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # Первое условие проверяет наличие предсказания в списке фактических элементов
        # второе условие - проверка на отсутствие (или наличие) повторов в предсказании
        if p in fact and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    return score / min(len(fact), n)

def MAP_at_n(fact, predicted, n=10):
    """
    Вычисление mean average precision at n для двух списков элементов.
   
    Вход
    ----------
    fact : list
            Фактический список элементов, который нужно предсказать.
    predicted : list
            Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
            Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Mean average precision at n двух списков
    """
    return np.mean([Avg_Precision_at_n(f,p,n) for f,p in zip(fact, predicted)])

In [3]:
#Загрузим данные из файлов. Для начала создадим два датасета - "products" и "transactions", соответственно:

data_dir = './data/recsys/'

PRODUCTS_CSV_PATH = os.path.join(data_dir, 'products.csv')
TRANSACTIONS_CSV_PATH = os.path.join(data_dir, 'transactions.csv')

products = pd.read_csv(PRODUCTS_CSV_PATH)
transactions = pd.read_csv(TRANSACTIONS_CSV_PATH)

#Теперь объединим датафреймы по product_id, исключив ранее обсуждаемые товары:
main = transactions.merge(products, how = 'left')

In [4]:
#Разделим данные на train и testset. Мне показалось логичным  в test отправить данные последнего заказа каждого пользователя.
testset = main[main.groupby('user_id')['order_number'].transform('max') == main['order_number']]
trainset = main[main.groupby('user_id')['order_number'].transform('max') != main['order_number']]
set_len = trainset.shape[0] + testset.shape[0]
print('\n К-во строк в тренировочном сете: ', trainset.shape[0], '\n','В тестовом сете ', testset.shape[0])
print('\n В процентом соотношении: ', round(100*(trainset.shape[0]/set_len),1), '% к ', 
      round(100*(testset.shape[0]/set_len),1), '%')


 К-во строк в тренировочном сете:  25328932 
 В тестовом сете  1079141

 В процентом соотношении:  95.9 % к  4.1 %


In [5]:
#Отрежем нужные нам столбцы из trainset:
main_sub = trainset[['order_id', 'user_id', 'order_number','product_id', 'add_to_cart_order','reordered']]
#Учтем номер заказа, посчитав соответсвующий коэффициент
main_sub['order_num_coef'] = main_sub['order_number'].apply(lambda x: round((np.log10(x) + 1),2))

/tmp/ipykernel_195143/563295715.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_sub['order_num_coef'] = main_sub['order_number'].apply(lambda x: round((np.log10(x) + 1),2))


Разберемся с учетом данных по порядку добавления в корзину ("add_to_cart_order").

Как уже говорилось ранее, чем раньше выбран продукт, тем его порядковый вес должен быть больше. Поэтому попросту "перевернем" все значения и отсечем всё, что было после 10-й позиции, как нерелевантное. Формула элементарна:

${(x_{max}+1) - x}$

Т.е. едля первой коэффициент будет 10 (11-1), для второй - 9 (11-2), для третьей - 8 (11-3). Для 10-й позиции, соответственно вес станет 1, а всё последующее попросту удалим.

In [6]:
#Вычислим искомый коэффициент:
main_sub['add_to_cart_coef'] = (main_sub.groupby('user_id')['add_to_cart_order'].transform('max') + 1)-main_sub['add_to_cart_order']
#Удалим значения ниже единицы:
main_sub= main_sub[main_sub.add_to_cart_coef > 0]
#Check
main_sub.head(15)

/tmp/ipykernel_195143/904859672.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_sub['add_to_cart_coef'] = (main_sub.groupby('user_id')['add_to_cart_order'].transform('max') + 1)-main_sub['add_to_cart_order']


,order_id,user_id,order_number,product_id,add_to_cart_order,reordered,order_num_coef,add_to_cart_coef
0,2539329,1,1,196,1.0,0.0,1.00,8.0
1,2539329,1,1,14084,2.0,0.0,1.00,7.0
2,2539329,1,1,12427,3.0,0.0,1.00,6.0
3,2539329,1,1,26088,4.0,0.0,1.00,5.0
4,2539329,1,1,26405,5.0,0.0,1.00,4.0
5,2398795,1,2,196,1.0,1.0,1.30,8.0
6,2398795,1,2,10258,2.0,0.0,1.30,7.0
7,2398795,1,2,12427,3.0,1.0,1.30,6.0
8,2398795,1,2,13176,4.0,0.0,1.30,5.0
9,2398795,1,2,26088,5.0,1.0,1.30,4.0


In [7]:
#Теперь скоректируем его с учетом номера заказа:
main_sub['add_to_cart_coef'] = main_sub['add_to_cart_coef'] * main_sub['order_num_coef']
#Le check
main_sub.head(15)

,order_id,user_id,order_number,product_id,add_to_cart_order,reordered,order_num_coef,add_to_cart_coef
0,2539329,1,1,196,1.0,0.0,1.00,8.00
1,2539329,1,1,14084,2.0,0.0,1.00,7.00
2,2539329,1,1,12427,3.0,0.0,1.00,6.00
3,2539329,1,1,26088,4.0,0.0,1.00,5.00
4,2539329,1,1,26405,5.0,0.0,1.00,4.00
5,2398795,1,2,196,1.0,1.0,1.30,10.40
6,2398795,1,2,10258,2.0,0.0,1.30,9.10
7,2398795,1,2,12427,3.0,1.0,1.30,7.80
8,2398795,1,2,13176,4.0,0.0,1.30,6.50
9,2398795,1,2,26088,5.0,1.0,1.30,5.20


In [8]:
#Посчитаем коэффициент перезаказов c учетом add_to_cart_coef:
main_sub['reordered'] = main_sub['reordered'] * main_sub['add_to_cart_coef']

In [9]:
# Посчитаем рейтинг, как add_to_cart_coef с учетом регулярности покупки продукта покупателем (reordered)
# Доли показателей определим как 0.8 и 0.2 в результирующем соответственно.
main_sub['rating']  = (main_sub.add_to_cart_coef * 0.8) + (main_sub.reordered * 0.2)
#Что получилось в итоге:
main_sub.head(15)

,order_id,user_id,order_number,product_id,add_to_cart_order,reordered,order_num_coef,add_to_cart_coef,rating
0,2539329,1,1,196,1.0,0.00,1.00,8.00,6.40
1,2539329,1,1,14084,2.0,0.00,1.00,7.00,5.60
2,2539329,1,1,12427,3.0,0.00,1.00,6.00,4.80
3,2539329,1,1,26088,4.0,0.00,1.00,5.00,4.00
4,2539329,1,1,26405,5.0,0.00,1.00,4.00,3.20
5,2398795,1,2,196,1.0,10.40,1.30,10.40,10.40
6,2398795,1,2,10258,2.0,0.00,1.30,9.10,7.28
7,2398795,1,2,12427,3.0,7.80,1.30,7.80,7.80
8,2398795,1,2,13176,4.0,0.00,1.30,6.50,5.20
9,2398795,1,2,26088,5.0,5.20,1.30,5.20,5.20


In [10]:
#Теперь уберём информацию о номере заказа, она нам больше не нужна.
#Для этого сгруппируем данные по пользователям и продуктам и просуммируем рейтинг:
coef_df = main_sub.groupby(['user_id','product_id'], as_index = False).agg(rating = ('rating','sum'))
#Check
coef_df.head(15)

,user_id,product_id,rating
0,1,196,107.130
1,1,10258,78.790
2,1,10326,5.440
3,1,12427,90.670
4,1,13032,10.520
5,1,13176,6.900
6,1,14084,5.600
7,1,17122,4.080
8,1,25133,61.770
9,1,26088,9.200


In [11]:
#Значения меньше одного нерелевантны для нас, поэтому удалим их:
coef_df = coef_df[coef_df.rating > 1]
coef_df['rating'].describe(percentiles=[.25, .5, .75, .90]).apply("{0:.5f}".format)

count    9133424.00000
mean         106.65921
std          243.09930
min            1.04000
25%           19.20000
50%           38.27200
75%           92.09000
90%          236.79000
max        16460.41000
Name: rating, dtype: object

In [12]:
#Scale from 1 to 10:
autoscaler = MinMaxScaler(feature_range = (1,10))
def sc(row):
    return autoscaler.fit_transform(row.values.reshape(-1,1))
coef_df['scaled'] = coef_df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)

/tmp/ipykernel_195143/749177484.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coef_df['scaled'] = coef_df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)


In [13]:
#Завершающие штрихи:
coef_df.drop(columns = ['rating'], inplace = True)
coef_df.rename(columns={"scaled": "rating"}, inplace = True)
coef_df['rating'] = coef_df['rating'].astype(np.float16)
coef_df.head(15)

,user_id,product_id,rating
0,1,196,10.000000
1,1,10258,7.558594
2,1,10326,1.234375
3,1,12427,8.578125
4,1,13032,1.671875
5,1,13176,1.360352
6,1,14084,1.248047
7,1,17122,1.117188
8,1,25133,6.089844
9,1,26088,1.558594


In [14]:
#Сохраним, чтобы пользоваться df из загруженного файла и не перезапускать постоянно долго соображающий код:
coef_df.to_csv('./data/recsys/fulltrain_df.csv', sep='\t', index=False, header = False)

In [15]:
#Также обработаем testset:
test_df = testset[["user_id","product_id","add_to_cart_order"]]
#Вычислим рейтинг простейшим образом - "перевернем" порядок покупок
test_df['rating'] = 11 -test_df['add_to_cart_order']
test_df['rating'] = test_df.rating.astype(np.int32)
#Scale from 1 to 10
test_df['rating'] = test_df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
test_df.drop(columns = ['add_to_cart_order'], inplace = True)
#Check
test_df.head(15)

/tmp/ipykernel_195143/1886979099.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['rating'] = 11 -test_df['add_to_cart_order']
/tmp/ipykernel_195143/1886979099.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['rating'] = test_df.rating.astype(np.int32)
/tmp/ipykernel_195143/1886979099.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (D

,user_id,product_id,rating
50,1,196,10.000
51,1,46149,8.875
52,1,39657,7.750
53,1,38928,6.625
54,1,25133,5.500
55,1,10258,4.375
56,1,35951,3.250
57,1,13032,2.125
58,1,12427,1.000
238,2,24852,10.000


In [16]:
# #Также обработаем testset:
# test_df = testset[["user_id","product_id","add_to_cart_order"]]
# #Вычислим рейтинг простейшим образом - "перевернем" порядок покупок
# test_df['rating'] = ((test_df.groupby('user_id')['add_to_cart_order'].transform('max') + 1) -test_df['add_to_cart_order'])
# test_df['rating'] = test_df.rating.astype(np.int32)
# #Scale from 1 to 10
# test_df['rating'] = test_df.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
# test_df.drop(columns = ['add_to_cart_order'], inplace = True)
# #Check
# test_df.head(15)

In [17]:
#Также сохраним:
test_df.to_csv('./data/recsys/fulltest_df.csv',sep='\t', index=False, header = False)

In [18]:
#Для подачи в модели:
files_dir = './data/recsys/'

TRAIN_CSV_PATH = os.path.join(files_dir, 'fulltrain_df.csv')
TEST_CSV_PATH = os.path.join(files_dir, 'fulltest_df.csv')

reader = Reader(line_format="user item rating", sep = '\t',rating_scale=(1, 10)) # Зададим разброс оценок

folds_files = [(TRAIN_CSV_PATH,TEST_CSV_PATH)] #список путей к файлам для подачи в объект библиотеки surprise lib
data = Dataset.load_from_folds(folds_files, reader=reader) #создадим data-объект

pkf = PredefinedKFold() #создадим объект, позволяющий подать в модель собственный набор train и test данных
trainset, testset = next(pkf.split(data)) #определим train и test сеты

In [19]:
#Установим модель с лучшим результатом:
bsl_options =  {'method': 'als', 'n_epochs': 20, 'reg_u': 18, 'reg_i': 6}
algo = BaselineOnly(bsl_options = bsl_options)

In [20]:
%%time
#Сделаем предсказание:
predictions = algo.fit(trainset).test(testset)

Estimating biases using als...
CPU times: user 15.6 s, sys: 39.5 ms, total: 15.6 s
Wall time: 15.6 s


In [21]:
#Соберем в сет:
appended_data = []
for i in predictions:
    appended_data.append(i)
pred_df = pd.DataFrame(appended_data, columns = ['user_id','product_id','real_rating','predicted_rating','details'])
pred_df.drop(columns = ["details"], inplace = True)

In [22]:
pred_df["user_id"] = pred_df.user_id.astype(np.int32)
pred_df["product_id"] = pred_df.product_id.astype(np.int32)
#Теперь нам нужно правильно отсортировать предсказание по столбцу с предполагаемым рейтингом, обрезав его до 10 значений:
train_trim = pred_df.merge(pred_df
        #Сгруппируем по пользователям, найдем 100 макс значений предсказанного рейтинга
        .groupby('user_id').predicted_rating.nlargest(20)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
#Проверка
train_trim.head(15)

,user_id,product_id,real_rating,predicted_rating
0,1,38928,6.625,4.109051
1,1,46149,8.875,4.043317
2,1,196,10.000,4.027638
3,1,35951,3.250,3.398892
4,1,39657,7.750,3.312071
5,1,12427,1.000,3.085271
6,1,25133,5.500,2.960512
7,1,10258,4.375,2.876630
8,1,13032,2.125,2.495094
9,2,24852,10.000,5.199750


In [23]:
X_train = train_trim[['user_id', 'product_id']].values
y_train = train_trim['real_rating'].values

n_factors = 25

X_train_array = [X_train[:, 0], X_train[:, 1]]

In [24]:
class EmbeddingLayer:
    def __init__(self, n_items, n_factors):
        self.n_items = n_items
        self.n_factors = n_factors
    
    def __call__(self, x):
        x = Embedding(self.n_items, self.n_factors, embeddings_initializer='he_normal', 
                      embeddings_regularizer=tf.keras.regularizers.l2(1e-6))(x)
        x = Reshape((self.n_factors,))(x)
        
        return x
    
def recommender_model(n_users, n_products, n_factors = 25, learning_rate =1e-3, min_rating = 1, max_rating = 10):
    user = Input(shape=(1,))
    u = EmbeddingLayer(n_users, n_factors)(user)
    ubias = EmbeddingLayer(n_users, 1)(user)
    
    product = Input(shape=(1,))
    p = EmbeddingLayer(n_products, n_factors)(product)
    pbias = EmbeddingLayer(n_products, 1)(product)   
    
    x = Dot(axes=1)([u, p])
    x = Add()([x, ubias, pbias])
    x = Dense(1, activation = 'sigmoid', kernel_initializer='lecun_uniform')(x)
    out = Lambda(lambda x: x * (max_rating - min_rating) + min_rating)(x)  
    
    model = Model(inputs=[user, product], outputs=out)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss=tf.keras.losses.MSE

    model.compile(loss=loss, optimizer=optimizer, metrics = ['accuracy']) 
    
    return model

In [25]:
n_users = train_trim.user_id.nunique()
n_products = train_trim.product_id.nunique()
rec_model = recommender_model(n_users, n_products)
rec_model.summary()

2025-02-28 22:26:16.204364: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.383151: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.383190: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.385837: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.385868: I external/local_xla/xla/stream_executor

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 1)]                  0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 1)]                  0         []                            
                                                                                                  
 embedding (Embedding)       (None, 1, 25)                2500000   ['input_1[0][0]']             
                                                                                                  
 embedding_2 (Embedding)     (None, 1, 25)                867975    ['input_2[0][0]']             
                                                                                              

/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.523730: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.523773: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 22:26:16.523778: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2022] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.
2025-02-28 22:26:16.523803: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000

In [26]:
#Создадим чекпойнт:
cpt_path ='checkpoints/instacart/als_checkpoint_rec.h5'
checkpoint = ModelCheckpoint(cpt_path, monitor='loss', verbose=1, save_best_only=True, mode='min', 
                                                save_freq = 'epoch')
stopper = EarlyStopping(monitor = 'loss', verbose = 1, restore_best_weights=True, mode="min", patience = 15)

In [27]:
#Обучим модель:
history = rec_model.fit(X_train_array, y_train, epochs=100, batch_size=64, callbacks=[checkpoint, stopper])

Epoch 1/100


2025-02-28 18:33:20.383512: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fe079e96540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-02-28 18:33:20.383535: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2025-02-28 18:33:20.396332: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-02-28 18:33:20.425734: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1740756800.478613  139373 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


15572/15576 [============================>.] - ETA: 0s - loss: 8.2024 - accuracy: 0.0966
Epoch 1: loss improved from inf to 8.20246, saving model to checkpoints/instacart/als_checkpoint_rec.h5
15576/15576 [==============================] - 64s 4ms/step - loss: 8.2025 - accuracy: 0.0966
Epoch 2/100
   27/15576 [..............................] - ETA: 1:02 - loss: 7.9096 - accuracy: 0.0868

/home/nette/miniconda3/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


15568/15576 [============================>.] - ETA: 0s - loss: 7.6650 - accuracy: 0.0966
Epoch 2: loss improved from 8.20246 to 7.66501, saving model to checkpoints/instacart/als_checkpoint_rec.h5
15576/15576 [==============================] - 63s 4ms/step - loss: 7.6650 - accuracy: 0.0966
Epoch 3/100
15568/15576 [============================>.] - ETA: 0s - loss: 6.6485 - accuracy: 0.0966
Epoch 3: loss improved from 7.66501 to 6.64872, saving model to checkpoints/instacart/als_checkpoint_rec.h5
15576/15576 [==============================] - 62s 4ms/step - loss: 6.6487 - accuracy: 0.0966
Epoch 4/100
15567/15576 [============================>.] - ETA: 0s - loss: 6.1861 - accuracy: 0.0966
Epoch 4: loss improved from 6.64872 to 6.18598, saving model to checkpoints/instacart/als_checkpoint_rec.h5
15576/15576 [==============================] - 62s 4ms/step - loss: 6.1860 - accuracy: 0.0966
Epoch 5/100
15572/15576 [============================>.] - ETA: 0s - loss: 6.0121 - accuracy: 0.0966
Ep

Epoch 30/100
15574/15576 [============================>.] - ETA: 0s - loss: 5.7996 - accuracy: 0.0966
Epoch 30: loss did not improve from 5.79676
15576/15576 [==============================] - 61s 4ms/step - loss: 5.7996 - accuracy: 0.0966
Epoch 31/100
15568/15576 [============================>.] - ETA: 0s - loss: 5.7988 - accuracy: 0.0966
Epoch 31: loss did not improve from 5.79676
15576/15576 [==============================] - 61s 4ms/step - loss: 5.7991 - accuracy: 0.0966
Epoch 32/100
15570/15576 [============================>.] - ETA: 0s - loss: 5.7984 - accuracy: 0.0966
Epoch 32: loss did not improve from 5.79676
15576/15576 [==============================] - 61s 4ms/step - loss: 5.7984 - accuracy: 0.0966
Epoch 33/100
15565/15576 [============================>.] - ETA: 0s - loss: 5.7986 - accuracy: 0.0966
Epoch 33: loss did not improve from 5.79676
15576/15576 [==============================] - 61s 4ms/step - loss: 5.7987 - accuracy: 0.0966
Epoch 34/100
15573/15576 [==============

In [26]:
#Загрузим модель из чекпойнта:
cpt_path ='checkpoints/instacart/als_checkpoint_rec.h5'
rec_model = tf.keras.models.load_model(cpt_path)

In [27]:
rec_preds = rec_model.predict(X_train_array)

31152/31152 [==============================] - 40s 1ms/step


In [28]:
train_trim['predicted'] = rec_preds
train_trim.head(15)

,user_id,product_id,real_rating,predicted_rating,predicted
0,1,38928,6.625,4.109051,6.134921
1,1,46149,8.875,4.043317,6.134921
2,1,196,10.000,4.027638,9.910060
3,1,35951,3.250,3.398892,6.134921
4,1,39657,7.750,3.312071,6.134921
5,1,12427,1.000,3.085271,1.304467
6,1,25133,5.500,2.960512,3.290379
7,1,10258,4.375,2.876630,3.201771
8,1,13032,2.125,2.495094,1.854682
9,2,24852,10.000,5.199750,9.890474


In [29]:
pred_result = train_trim.merge(train_trim
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').predicted.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся

#Проверка
pred_result.head(15)

,user_id,product_id,real_rating,predicted_rating,predicted
0,1,196,10.000,4.027638,9.910060
1,1,38928,6.625,4.109051,6.134921
2,1,46149,8.875,4.043317,6.134921
3,1,35951,3.250,3.398892,6.134921
4,1,39657,7.750,3.312071,6.134921
5,1,38928,6.625,4.109051,6.134921
6,1,46149,8.875,4.043317,6.134921
7,1,35951,3.250,3.398892,6.134921
8,1,39657,7.750,3.312071,6.134921
9,1,38928,6.625,4.109051,6.134921


In [30]:
#Соберем все id продуктов в один столбец - pred_order:
pred_result = pred_result.groupby('user_id')['product_id'].unique().reset_index()
pred_result.columns=['user_id','pred_order']

In [31]:
#Таким же образом соберем факт:
fact_df = train_trim[['user_id','product_id','real_rating']]
fact_df = fact_df.merge(fact_df
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').real_rating.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
fact_df.drop_duplicates(inplace = True)
fact_df = fact_df.groupby('user_id')['product_id'].unique().reset_index()
fact_df.columns=['user_id','fact_order']
#Check
fact_df.head(15)

,user_id,fact_order
0,1,"[196, 46149, 39657, 38928, 25133, 10258, 35951..."
1,2,"[24852, 16589, 1559, 19156, 18523, 22825, 2741..."
2,3,"[39190, 47766, 21903, 43961, 17668]"
3,7,"[47272, 29993, 31683, 27690, 9598, 13198, 3039..."
4,13,"[27435, 27086, 4210, 43086, 34382, 41926, 1419..."
5,14,"[39399, 29509, 23803, 15869, 8744, 37266, 1113..."
6,15,"[196, 48142]"
7,17,"[7350, 18534, 9006, 26767, 14146, 16797, 18567..."
8,21,"[25740, 6576, 17982, 25513, 8214, 18523, 24799..."
9,22,"[35221, 24964, 7948, 24506]"


In [32]:
#Посмотрим на результат:
result = fact_df.merge(pred_result, how = 'left')
result['fact_order'] = result['fact_order'].astype(str)
result['pred_order'] = result['pred_order'].astype(str) 
#Применим функцию MAP_at_n построчно:
result['MAP'] = result.apply(lambda x: MAP_at_n(x.fact_order, x.pred_order), axis=1)
result.head()

,user_id,fact_order,pred_order,MAP
0,1,[ 196 46149 39657 38928 25133 10258 35951 130...,[ 196 38928 46149 35951 39657 25133 10258 130...,0.509091
1,2,[24852 16589 1559 19156 18523 22825 27413 337...,[24852 16589 19156 1559 21709 27413 18523 77...,0.508197
2,3,[39190 47766 21903 43961 17668],[47766 39190 43961 21903 17668],0.419355
3,7,[47272 29993 31683 27690 9598 13198 30391 376...,[29993 31683 27690 9598 30391 13198 21137 408...,0.377049
4,13,[27435 27086 4210 43086 34382 41926 14197 42248],[27086 27435 4210 43086 41926 42248 34382 14197],0.510204


In [33]:
sum(result.MAP)/len(result.user_id.unique())

0.5141695687527286

In [34]:
#Проверим точность на этом предсказании в Kaggle:
pred_grouped = train_trim.merge(train_trim
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').predicted.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
pred_grouped.drop_duplicates(inplace = True)
pred_grouped['product_id'] = pred_grouped['product_id'].astype(str)
prediction_df = pred_grouped.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
prediction_df.to_csv('./data/recsys/prediction_als_nn.csv',sep=',', index=False)